# ECE1512 Project A - Build an ABMIL training pipeline in PyTorch

This notebook uses pre-extracted whole-slide histopathology features and guides you through a complete PyTorch classification pipeline. The goal is not merely to run ABMIL; it is to understand the code path from data loading to optimization, validation, checkpoint selection, and final testing.

Please complete every cell marked `TODO`. Dataset engineering and metric boilerplate are provided so you can focus on the machine-learning training workflow.

This project requires basic familiarity with Python object-oriented programming (OOP). If you have limited experience with Python OOP, please review the provided introductory video before starting the project. 

[PyTorch](https://docs.pytorch.org/tutorials/beginner/introyt/introyt1_tutorial.html)

[Python Object-Oriented Programming ](https://www.youtube.com/watch?v=JeznW_7DlB0)

**Research extension:** Propose two research ideas and implement and evaluate one selected idea. Report ABMIL baseline and selected-extension results on **CAMELYON16, CAMELYON17, and BRACS**, using the same features and train/validation/test splits within each dataset. 


## Learning objectives

After this project you should be able to explain and implement:

1. `Dataset` vs. `DataLoader`, batching, shuffling.
2. `nn.Module`, trainable parameters, and `forward()`.
3. A classification loss and optimizer.
4. The training sequence `zero_grad -> forward -> loss -> backward -> step`.
5. The difference between `model.train()` and `model.eval()`.

## Step 0. Download Pre-extracted WSI Features & Select the Project Directory
Download the required feature file(s) from Zenodo or Hugging Face:

https://zenodo.org/records/23130543

https://huggingface.co/datasets/LeonYe/ECE1512-WSI-Feature


Place the HDF5 file under the path expected by your YAML configuration in the `config/`, or edit data_dir in the YAML file.


In [ ]:
!pip install -q torchmetrics h5py pandas scikit-learn pyyaml matplotlib openslide-python openslide-bin


In [ ]:
import os, sys, copy, time
from pathlib import Path
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from utils import TODO


If you are using Google Colab with Drive, mount Drive first and change `PROJECT_DIR`.

In [ ]:
# # Optional in Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')

PROJECT_DIR = './'
DATASET = 'C16'  # Choose: 'C16', 'C17', or 'BRACS'

CONFIG_FILES = {
    'C16': 'camelyon16_medical_ssl_config.yml',
    'C17': 'camelyon17_medical_ssl_config.yml',
    'BRACS': 'bracs_medical_ssl_config.yml',
}
CONFIG_PATH = os.path.join(PROJECT_DIR, 'config', CONFIG_FILES[DATASET])

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print('Project:', os.path.abspath(PROJECT_DIR))
print('Config :', CONFIG_PATH)

from wsi_visualization import visualize_wsi_attention


In [ ]:
from dataloader import build_HDF5_feat_dataset, mil_collate_fn
from utils import load_config, seed_everything, infer_feature_dim, classification_metrics

conf = load_config(CONFIG_PATH)
seed_everything(int(conf.seed))
print(conf)

if os.path.exists(conf.data_dir):
    print('Feature file exists:', os.path.exists(conf.data_dir), conf.data_dir)
else:
    print('\nDownload the required feature file from: https://zenodo.org/records/23130543')
    print('Then update data_dir in:', CONFIG_PATH)
    

# Step 1 - Dataset and DataLoader

`build_HDF5_feat_dataset` is provided. It creates train/validation/test `Dataset` objects without loading all slide features into memory at once.


Complete the three `DataLoader` definitions. Think about which split should be shuffled.

[Download WSI datasets](https://zenodo.org/records/23130543)

[Datasets & DataLoaders](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html)


[PyTorch DataLoader](https://docs.pytorch.org/docs/2.14/data.html#torch.utils.data.DataLoader)


[PyTorch Datasets](https://docs.pytorch.org/vision/stable/datasets.html)


In [ ]:
train_data, val_data, test_data = build_HDF5_feat_dataset(conf.data_dir, conf)
conf.D_feat = infer_feature_dim(train_data)
print('Split sizes: Train {}, Validation {}, Test {};'.format(len(train_data), len(val_data), len(test_data)))
print("Feature Shape: {};".format(tuple(train_data[1]['input'].shape)))
print("Label: {}; \nSlide ID: {}.".format(train_data[1]['label'], train_data[1]['slide_id']))

In [ ]:
batch_size=conf.B
num_workers=int(conf.n_worker)
pin_memory=bool(conf.pin_memory) and torch.cuda.is_available()
collate_fn=mil_collate_fn
print("DataLoader args: Batch size {}; \n \
      # CPUs {};  \n \
      pin_memory {};  \n \
      collate_fn {}".format(batch_size, num_workers, pin_memory, collate_fn))


In [ ]:
# TODO 1A: create the training DataLoader. Training data should be shuffled.
train_loader = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True,              # new slide order every epoch
    num_workers=num_workers,
    pin_memory=pin_memory,
    collate_fn=collate_fn,     # pads variable-size bags and returns a patch mask
)

# TODO 1B: create validation and test DataLoaders. Do not shuffle them.
val_loader = DataLoader(
    val_data,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    collate_fn=collate_fn,
)
test_loader = DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    collate_fn=collate_fn,
)

In [ ]:
# Inspect one MIL batch.
batch = next(iter(train_loader))
print('input:', tuple(batch['input'].shape), batch['input'].dtype)
print('label:', tuple(batch['label'].shape), batch['label'].dtype)
print('slide IDs:', batch['slide_id'])

**Questions for your report:**

- What is the difference between a `Dataset` and a `DataLoader`?
- Which Dataloader should use `shuffle=True`? Why?
- For a batch with input shape `[B, N, D]`, what do `B`, `N`, and `D` mean in this project?


**Answers.**

- A `Dataset` defines *what one sample is*: `__len__` and `__getitem__(i)`, which here lazily reads one slide (a bag of patch features `[N, D]`, its label and slide ID) from the HDF5 file. A `DataLoader` defines *how samples are served during training*: it draws indices (optionally shuffled), groups them into batches with `collate_fn` (here: zero-padding bags to the same `N` and returning a boolean `mask`), and can use worker processes and pinned memory.
- Only the **training** loader is shuffled. Stochastic optimisation assumes roughly i.i.d. mini-batches; a fixed order (e.g. the order in which slide IDs are listed) gives correlated consecutive updates and the same ordering bias every epoch. Validation/test metrics do not depend on order, and a fixed order keeps predictions aligned with slide IDs and runs reproducible.
- `B` is the number of slides (bags) in a batch (`B = 1` here), `N` is the number of patches (instances) in the bag, which differs per slide (padded to the largest bag when `B > 1`), and `D` is the dimension of each pre-extracted patch embedding (`D = 384`, from the frozen self-supervised ViT-S/16 encoder).

# Step 2 - Implement the ABMIL model

For each patch feature $h_i$, the gated attention score follows Ilse et al. (2018):

$$
a_i = w^T\left[\tanh(Vh_i) \odot \sigma(Uh_i)\right].
$$

The attention logits are normalized over instances:

$$
\alpha_i = \frac{\exp(a_i)}{\sum_j \exp(a_j)},
$$

and the slide representation is

$$
z = \sum_i \alpha_i h_i.
$$

You will implement the learnable layers and the key `forward()` operations.

[Custom nn Modules](https://docs.pytorch.org/tutorials/beginner/examples_nn/polynomial_module.html)

[nn.Linear](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Linear.html)

[nn.modules.dropout](https://docs.pytorch.org/docs/2.14/generated/torch.nn.Dropout.html)

[nn.ReLU](https://docs.pytorch.org/docs/2.14/generated/torch.nn.ReLU.html)

[torch.nn.functional.softmax (F.softmax)](https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.softmax.html)

[torch.bmm](https://docs.pytorch.org/docs/2.14/generated/torch.bmm.html)

In [ ]:
class Classifier1FC(nn.Module):
    def __init__(self, n_channels, n_classes, droprate=0.0):
        super().__init__()
        self.dropout = nn.Dropout(droprate) if droprate > 0 else nn.Identity()
        # TODO 2A: one linear layer from n_channels to n_classes.
        self.fc = nn.Linear(n_channels, n_classes)

    def forward(self, x):
        # TODO 2B: apply dropout, then the classifier.
        return self.fc(self.dropout(x))


class DimReduction(nn.Module):
    def __init__(self, n_channels, m_dim):
        super().__init__()
        # TODO 2C: project patch features to m_dim and use ReLU.
        self.fc = nn.Linear(n_channels, m_dim, bias=False)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # TODO 2D: linear projection followed by ReLU.
        return self.relu(self.fc(x))


class AttentionGated(nn.Module):
    def __init__(self, L=256, D=128, K=1):
        super().__init__()
        self.attention_V = nn.Sequential(nn.Linear(L, D), nn.Tanh())
        self.attention_U = nn.Sequential(nn.Linear(L, D), nn.Sigmoid())
        self.attention_weights_w = nn.Linear(D, K)

    def forward(self, x):
        # x: [B, N, L]
        # TODO 2E: implement the two gated branches.
        a_v = self.attention_V(x)            # tanh(V h_i)     [B, N, D]
        a_u = self.attention_U(x)            # sigmoid(U h_i)  [B, N, D]

        a = self.attention_weights_w(a_v * a_u)   # [B, N, K]
        return a.transpose(1, 2)                  # [B, K, N]


class ABMIL(nn.Module):
    def __init__(self, conf):
        super().__init__()
        self.dimreduction = DimReduction(conf.D_feat, conf.D_inner)
        self.attention = AttentionGated(conf.D_inner, conf.D_attn, K=1)
        self.classifier = Classifier1FC(conf.D_inner, conf.n_class, conf.droprate)

    def forward(self, x, mask=None, return_attention=False):
        # x: [B, N, D_feat]; mask: optional [B, N] bool from mil_collate_fn (True = real patch)
        med_feat = self.dimreduction(x)       # [B, N, D_inner]
        a_logits = self.attention(med_feat)   # [B, 1, N]
        if mask is not None:                  # padded patches must get zero attention (needed when B > 1)
            a_logits = a_logits.masked_fill(~mask[:, None, :], float('-inf'))

        # TODO 2G: normalize attention logits over the instance dimension.
        attention = F.softmax(a_logits, dim=-1)                # [B, 1, N], sums to 1 over N

        # TODO 2H: weighted aggregation. torch.bmm is useful here.
        bag_feat = torch.bmm(attention, med_feat).squeeze(1)   # [B, D_inner]

        # TODO 2I: classify the slide representation.
        logits = self.classifier(bag_feat)                     # [B, n_class]

        if return_attention:
            return logits, attention
        return logits

**Sanity Check**

Before running the full training pipeline, check the tensor shapes and attention normalization.


In [ ]:
model_cpu = ABMIL(conf).eval()

x_demo = torch.randn(2, 7, conf.D_feat)


with torch.no_grad():
    logits_demo, attn_demo = model_cpu(x_demo, return_attention=True)

print('logits:', logits_demo.shape)
print('attention:', attn_demo.shape)
print('attention sums:', attn_demo.sum(dim=-1))

assert logits_demo.shape == (2, int(conf.n_class))
assert attn_demo.shape == (2, 1, 7)
torch.testing.assert_close(attn_demo.sum(dim=-1), torch.ones(2, 1))
print('Sanity check passed.')


**Questions:**

- Which tensors in your model contain trainable parameters?
- What properties do attention weights have after softmax?
- Could we replace softmax with another function?


**Answers.**

- Trainable parameters (all `nn.Linear` weights/biases): `dimreduction.fc.weight` (384→256, no bias), `attention.attention_V.0.{weight,bias}` and `attention.attention_U.0.{weight,bias}` (256→128 each), `attention.attention_weights_w.{weight,bias}` (128→1) and `classifier.fc.{weight,bias}` (256→C). The patch features, attention weights α and the bag feature `z` are activations, not parameters; the patch encoder that produced the 384-d features is frozen and not part of this model. (Run the cell below `print(sum(...))` in Step 3 for the count.)
- After softmax the weights are strictly positive, sum to 1 over the patches of the bag, and are permutation-equivariant (re-ordering patches re-orders α), so `z = Σ α_i h_i` is a convex combination of the instance embeddings and the whole model is permutation-invariant. Softmax is invariant to adding a constant to all logits, it is dense (no exact zeros), and its weights shrink as `N` grows, so on bags with thousands of patches most weights are tiny and a few can dominate.
- Yes. Any non-negative normalisation that is permutation-invariant works: a temperature-scaled softmax, `sparsemax`/`entmax` (exact zeros, sparser maps), sigmoid gates normalised by their sum, or hard choices (top-k, max = max pooling) and the uniform weighting `1/N` (mean pooling). Un-normalised sigmoid weights are also possible but make `z` scale with the number of patches, which differs strongly between slides.

# Step 3 - Device, loss, and optimizer

For multi-class slide classification, use cross-entropy as the baseline objective and AdamW as the baseline optimizer. Complete the code below.

[nn.CrossEntropyLoss](https://docs.pytorch.org/docs/2.14/generated/torch.nn.CrossEntropyLoss.html)

[torch.optim.AdamW](https://docs.pytorch.org/docs/2.14/generated/torch.optim.AdamW.html)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ABMIL(conf).to(device)

# TODO 3A: define the multi-class classification loss.
criterion = nn.CrossEntropyLoss()   # expects raw logits [B, C] and integer labels [B]

# TODO 3B: create AdamW and pass the model parameters, learning rate, and weight decay.
optimizer = torch.optim.AdamW(model.parameters(), lr=float(conf.lr), weight_decay=float(conf.wd))

print('Device:', device)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))
for name, p in model.named_parameters():
    print(f'  {name:40s} {tuple(p.shape)}')

# Step 4 - Complete one training epoch

The essential PyTorch optimization sequence is:

`model.train()` -> move tensors to device -> `zero_grad()` -> forward pass -> compute loss -> `backward()` -> `step()`.

Implement this sequence yourself. This is one of the main learning objectives of the project.

[Training with PyTorch](https://docs.pytorch.org/tutorials/beginner/introyt/trainingyt.html)

[train() vs eval()](https://discuss.pytorch.org/t/model-train-and-model-eval-vs-model-and-model-eval/5744)

In [ ]:
def train_one_epoch(model, criterion, data_loader, optimizer, device):
    # TODO 4A: put the model in training mode.
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for data in data_loader:
        # TODO 4B: move input and labels to the selected device.
        bag = data["input"].to(device, dtype=torch.float32)
        labels = data["label"].to(device, dtype=torch.long)
        mask = data["mask"].to(device)

        # TODO 4C: clear gradients from the previous iteration.
        optimizer.zero_grad()

        # TODO 4D: forward pass.
        logits = model(bag, mask=mask)

        # TODO 4E: compute the loss.
        loss = criterion(logits, labels)

        # TODO 4F: back-propagate gradients.
        loss.backward()

        # TODO 4G: update the model parameters.
        optimizer.step()

        batch_size = labels.shape[0]
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += batch_size

    return {
        'loss': total_loss / total_examples,
        'acc': 100.0 * total_correct / total_examples,
    }

# Step 5 - Validation and test evaluation

Evaluation should not update the model. Complete the missing lines and use the provided metric helper for macro-AUROC and macro-F1.


In [ ]:
@torch.no_grad()
def evaluate(model, criterion, data_loader, device, conf):
    # TODO 5A: switch to evaluation mode.
    model.eval()

    total_loss = 0.0
    total_examples = 0
    probabilities_all = []
    labels_all = []

    for data in data_loader:
        # TODO 5B: move data to device.
        bag = data["input"].to(device, dtype=torch.float32)
        labels = data["label"].to(device, dtype=torch.long)
        mask = data["mask"].to(device)

        # TODO 5C: forward pass and loss.
        logits = model(bag, mask=mask)
        loss = criterion(logits, labels)

        # TODO 5D: convert class logits into probabilities.
        probabilities = F.softmax(logits, dim=1)

        total_loss += loss.item() * labels.shape[0]
        total_examples += labels.shape[0]
        probabilities_all.append(probabilities)
        labels_all.append(labels)

    probabilities_all = torch.cat(probabilities_all, dim=0)
    labels_all = torch.cat(labels_all, dim=0)
    metrics = classification_metrics(probabilities_all, labels_all, conf.n_class)
    metrics['loss'] = total_loss / total_examples
    return metrics

**Checkpoint questions:**

- Why are `model.eval()` and `torch.no_grad()` different concepts?
- What would happen if gradients were accumulated because `optimizer.zero_grad()` was omitted?
- Why must the test set not be used to choose the best epoch?


**Answers.**

- `model.eval()` changes how layers *compute*: dropout is switched off and batch-norm would use running statistics (and in the Step 9 extension it also switches off the random top-k instance masking). `torch.no_grad()` changes whether autograd *records* the computation: no graph is stored, which saves memory and time but does not change any layer. Evaluation needs both: `eval()` alone still builds graphs, `no_grad()` alone still applies dropout and gives noisy metrics.
- `.grad` would keep accumulating: the update at step *t* would use the sum of all gradients since the start, so the effective step grows, stale directions dominate and training becomes unstable or diverges. (Accumulation is only useful on purpose, e.g. to simulate a larger batch, with `zero_grad()` every *k* steps.)
- Choosing the epoch on the test set uses test labels for a modelling decision. The selected checkpoint is then partly fitted to the noise of that particular test set, so the reported number is optimistically biased and no longer an estimate of performance on unseen slides. With small test sets (129 CAMELYON16, 200 CAMELYON17, 86 BRACS slides) this bias can be several AUC points. The validation set makes all decisions; the test set is used once at the end.

# Step 6 - Train, validate, and select the best checkpoint

Use the validation set to choose the best epoch. The test set must remain untouched until model selection is complete.

For debugging, set `NUM_EPOCHS = 2`. For your reported baseline, use the full configured epoch count unless instructed otherwise.


In [ ]:
NUM_EPOCHS = int(conf.train_epoch)  # You may temporarily set this to 2 while debugging.

best_score = float('-inf')
best_epoch = -1
best_model_state = None
history = []

for epoch in range(NUM_EPOCHS):
    start = time.time()
    train_metrics = train_one_epoch(model, criterion, train_loader, optimizer, device)
    val_metrics = evaluate(model, criterion, val_loader, device, conf)

    # Baseline selection criterion: macro-AUROC + macro-F1.
    score = val_metrics['auc'] + val_metrics['f1']

    if score > best_score:
        best_score = score
        best_epoch = epoch
        best_model_state = copy.deepcopy(model.state_dict())
    row = {
        'epoch': epoch,
        'train_loss': train_metrics['loss'],
        'train_acc': train_metrics['acc'],
        'val_loss': val_metrics['loss'],
        'val_acc': val_metrics['acc'],
        'val_auc': val_metrics['auc'],
        'val_f1': val_metrics['f1'],
    }
    history.append(row)
    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
          f"train loss={row['train_loss']:.4f} acc={row['train_acc']:.2f} | "
          f"val loss={row['val_loss']:.4f} auc={row['val_auc']:.3f} f1={row['val_f1']:.3f} | "
          f"best={best_epoch+1} | {time.time()-start:.1f}s")

assert best_model_state is not None


In [ ]:
# Plot learning curves.
epochs = [x['epoch'] + 1 for x in history]
plt.figure(figsize=(7, 4))
plt.plot(epochs, [x['train_loss'] for x in history], label='train loss')
plt.plot(epochs, [x['val_loss'] for x in history], label='validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.2)
plt.show()


# Step 7 - Final held-out test evaluation

Only now, after the best checkpoint has been chosen from validation performance, load it and evaluate once on the test set.


In [ ]:
model.load_state_dict(best_model_state)
test_metrics = evaluate(model, criterion, test_loader, device, conf)

print('Best validation epoch:', best_epoch + 1)
print('Final test metrics:')
for key, value in test_metrics.items():
    print(f'  {key}: {value:.4f}')


In [ ]:
# Save the single-run baseline of this notebook (the multi-seed numbers in the report come from run_experiments.py).
import json
os.makedirs(os.path.join(PROJECT_DIR, 'results', 'notebook'), exist_ok=True)
with open(os.path.join(PROJECT_DIR, 'results', 'notebook', f'{DATASET}_abmil.json'), 'w') as f:
    json.dump({'dataset': DATASET, 'best_epoch': best_epoch + 1, 'test': test_metrics, 'history': history}, f, indent=1)

# Step 8 -  Inspect Attention on One Slide

Attention is useful for understanding which instances contribute strongly to the learned bag representation, but attention weights should not automatically be interpreted as causal explanations.


In [ ]:
one_batch = mil_collate_fn([test_data[0]])
model.eval()
with torch.no_grad():
    logits, attention = model(
        one_batch['input'].to(device, dtype=torch.float32),
        return_attention=True,
    )

print('Slide:', one_batch['slide_id'][0])
print('Predicted class:', int(logits.argmax(dim=1)[0]))
print('Attention shape:', tuple(attention.shape))
print('Attention sum for first slide:', float(attention[0].sum()))


## Visualize Attention on Whole-Slide Images (WSIs)

Before running the attention visualization, please complete the following steps:

1. Visit the [ECE1512-WSI-Feature dataset on Hugging Face](https://huggingface.co/datasets/LeonYe/ECE1512-WSI-Feature)

2. Download the original WSIs `test_067.tif` and `test_068.tif` and the corresponding tumor annotation file `test_068.xml`.

3. Create a folder named `slide/` under your project directory and place the downloaded files in this folder.

Run this cell after training and final test evaluation, with `DATASET = 'C16'`. It restores `best_model_state` and plots the two slides in `PROJECT_DIR/slide`:

- **test_068 (tumor):** attention overlay and green contours from `test_068.xml`, including a zoom of the annotated region.
- **test_067 (normal):** attention overlay without annotation contours, including a zoom around the highest-attention patch.

The blue rectangle marks the zoom shown on the right. Colors show **attention percentiles within each slide**, not patch tumor probabilities; areas without extracted patches stay transparent. The displayed `P(tumor)` is the model's slide-level prediction.

HDF5 patch coordinates and XML contour coordinates are assumed to be expressed in level-0 `(x, y)`  coordinates; patch coordinates are treated as top-left corners. The function parameter `patch_size_level0=512` is an editable **assumption** for the patch footprint because these HDF5 files do not store patch-size metadata. Match it to the feature extractor's patch size for an exact footprint. All visualization functions are in `wsi_visualization.py`; the cell below only calls `visualize_wsi_attention`.

[OpenSlide Python](https://openslide.org/api/python/) reads a thumbnail and a pyramid-level crop rather than loading the full-resolution WSI into memory. If dependencies are missing, run `%pip install openslide-python openslide-bin` and restart the kernel if needed. Raw per-patch attention and coordinates are retained in `attention_visualizations`.


In [ ]:
from contextlib import contextmanager

@contextmanager
def save_shown_figures(paths):
    """visualize_wsi_attention calls plt.show() once per slide; also save each figure, in order, to `paths`."""
    original_show, queue = plt.show, list(paths)
    def show_and_save(*args, **kwargs):
        if queue:
            path = queue.pop(0)
            os.makedirs(os.path.dirname(path), exist_ok=True)
            plt.gcf().savefig(path, dpi=200, bbox_inches='tight')
        return original_show(*args, **kwargs)
    plt.show = show_and_save
    try:
        yield
    finally:
        plt.show = original_show

FIG_DIR = os.path.join(PROJECT_DIR, 'report', 'figures')
if DATASET == 'C16':
    with save_shown_figures([os.path.join(FIG_DIR, 'wsi_attention_test_068.png'),
                             os.path.join(FIG_DIR, 'wsi_attention_test_067.png')]):
        attention_visualizations = visualize_wsi_attention(model, conf, PROJECT_DIR, best_model_state)

# Step 9 - Research extension: propose two ideas, implement one of them

After completing the ABMIL baseline, propose **two distinct research ideas**. Select **one** for implementation and evaluation on **CAMELYON16 (C16), CAMELYON17 (C17), and BRACS**. You may explore explainability, more advanced model structures, improved objectives beyond cross-entropy, or another well-motivated direction.

**A negative result will not be penalized if the design choices, experiments, observations, and possible reasons for failure are well justified.** Report evidence relevant to your hypothesis: an explainability idea, for example, needs an explanation evaluation in addition to classification results.

Feel free to contact your TA to discuss your research ideas.

## Two research ideas

**Idea 1 (implemented): attention-challenging MIL — multi-branch attention + stochastic top-k instance masking** (re-implemented after Zhang et al., ECCV 2024).
With only a few hundred training slides, ABMIL tends to put most of its attention mass on a handful of very salient patches and fits the training set from those patterns (attention over-concentration), which hurts generalisation. The extension keeps the ABMIL encoder and gated attention but
(i) uses `K = 5` attention branches, each with its own classifier and cross-entropy loss, plus a diversity loss (mean pairwise cosine similarity of the branches' attention maps) so the branches look at different patches;
(ii) during training only, randomly hides 60% of each branch's top-10 attended patches before the softmax, so the model must find additional evidence.
The prediction comes from a slide classifier on the feature pooled with the branch-averaged attention; inference is deterministic. *Hypothesis:* attention becomes less concentrated (higher entropy, lower top-1% mass) and test AUC/F1 improve, most where training data are scarce. Ablations: MBA only (`acmil_mba`) and STKIM only (`acmil_stkim`).

**Idea 2 (proposed, not implemented): intrinsically interpretable Additive MIL with a faithfulness evaluation** (after Javed et al., NeurIPS 2022).
Attention weights say which patches are pooled, not how each patch pushes each class. Additive MIL moves the classifier before pooling, `logit_c = Σ_i α_i · f_c(h_i)`, so every patch has an exact, signed, class-wise contribution. It would be evaluated with (a) classification metrics against ABMIL, (b) deletion/insertion curves (how fast the predicted probability changes when the most important patches are removed/added) for attention vs. additive contributions, and (c) on CAMELYON16 the overlap between high-contribution patches and the tumour annotations.

See the report for the full motivation and design rationale. The cells below run Idea 1 on the dataset selected in this notebook with exactly the baseline protocol; `run_experiments.py` runs it on all three datasets with several seeds.

In [ ]:
from models import ACMIL
from engine import attention_statistics


def train_one_epoch_ext(model, criterion, data_loader, optimizer, device):
    """Same optimisation sequence as Step 4. The only difference is the objective,
    returned by model.training_loss: CE(slide head) + mean CE(branch heads) + diversity."""
    model.train()                                   # also enables stochastic top-k masking
    total_loss, total_correct, total_examples = 0.0, 0, 0
    for data in data_loader:
        bag = data["input"].to(device, dtype=torch.float32)
        labels = data["label"].to(device, dtype=torch.long)
        mask = data["mask"].to(device)
        optimizer.zero_grad()
        loss, logits = model.training_loss(bag, labels, criterion, mask=mask)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.shape[0]
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += labels.shape[0]
    return {'loss': total_loss / total_examples, 'acc': 100.0 * total_correct / total_examples}


# Same seed, optimiser, epochs and model-selection rule as the baseline.
seed_everything(int(conf.seed))
acmil = ACMIL(conf, n_branch=5, n_masked_patch=10, mask_drop=0.6, div_weight=1.0).to(device)
optimizer_ext = torch.optim.AdamW(acmil.parameters(), lr=float(conf.lr), weight_decay=float(conf.wd))
print('ACMIL trainable parameters:', sum(p.numel() for p in acmil.parameters() if p.requires_grad))

best_score_ext, best_epoch_ext, acmil_best_state, history_ext = float('-inf'), -1, None, []
for epoch in range(NUM_EPOCHS):
    start = time.time()
    tr = train_one_epoch_ext(acmil, criterion, train_loader, optimizer_ext, device)
    va = evaluate(acmil, criterion, val_loader, device, conf)   # Step 5 evaluate works unchanged
    score = va['auc'] + va['f1']
    if score > best_score_ext:
        best_score_ext, best_epoch_ext = score, epoch
        acmil_best_state = copy.deepcopy(acmil.state_dict())
    history_ext.append({'epoch': epoch, 'train_loss': tr['loss'], 'train_acc': tr['acc'],
                        'val_loss': va['loss'], 'val_acc': va['acc'], 'val_auc': va['auc'], 'val_f1': va['f1']})
    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | train objective={tr['loss']:.4f} acc={tr['acc']:.2f} | "
          f"val loss={va['loss']:.4f} auc={va['auc']:.3f} f1={va['f1']:.3f} | best={best_epoch_ext+1} | {time.time()-start:.1f}s")

In [ ]:
# Held-out test evaluation of the selected ACMIL checkpoint, next to the baseline from Step 7.
acmil.load_state_dict(acmil_best_state)
test_metrics_ext = evaluate(acmil, criterion, test_loader, device, conf)
model.load_state_dict(best_model_state)

print(f"{'':10s}{'AUC':>8s}{'ACC':>8s}{'F1':>8s}{'loss':>8s}  best epoch")
for name, m, be in [('ABMIL', test_metrics, best_epoch), ('ACMIL', test_metrics_ext, best_epoch_ext)]:
    print(f"{name:10s}{m['auc']:8.4f}{m['acc']:8.2f}{m['f1']:8.4f}{m['loss']:8.4f}  {be+1}")

# Validation loss is plain slide-level cross-entropy for both models, so these curves are comparable.
# (ACMIL's training objective also contains the branch and diversity terms, so it is not plotted against ABMIL's.)
plt.figure(figsize=(7, 4))
plt.plot(epochs, [x['val_loss'] for x in history], label='ABMIL validation loss')
plt.plot(epochs, [x['val_loss'] for x in history_ext], label='ACMIL validation loss')
plt.xlabel('Epoch'); plt.ylabel('Cross-entropy'); plt.legend(); plt.grid(alpha=0.2); plt.show()

In [ ]:
# Hypothesis check: is ACMIL's attention less concentrated than ABMIL's on the test slides?
import numpy as np

@torch.no_grad()
def attention_concentration(m, loader):
    m.eval()
    rows = []
    for data in loader:
        logits, attn = m(data["input"].to(device, dtype=torch.float32), mask=data["mask"].to(device),
                         return_attention=True)
        for b in range(attn.shape[0]):
            rows.append({**attention_statistics(attn[b, 0][data["mask"][b].to(device)].cpu()),
                         'label': int(data["label"][b])})
    return rows

conc = {'ABMIL': attention_concentration(model, test_loader), 'ACMIL': attention_concentration(acmil, test_loader)}
print(f"{'':8s}{'entropy/logN':>14s}{'eff. frac':>11s}{'top-1% mass':>13s}{'max weight':>12s}")
for name, rows in conc.items():
    print(f"{name:8s}" + "".join(f"{np.mean([r[k] for r in rows]):{w}.4f}"
          for k, w in [('entropy_norm', 14), ('eff_frac', 11), ('top1pct_mass', 13), ('max_weight', 12)]))

plt.figure(figsize=(6, 3.5))
for name, rows in conc.items():
    plt.hist([r['entropy_norm'] for r in rows], bins=20, alpha=0.6, label=name)
plt.xlabel('Normalized attention entropy per test slide'); plt.ylabel('# slides'); plt.legend(); plt.show()

In [ ]:
# CAMELYON16 only: the same WSI attention overlay for the extension (requires slide/test_067.tif, test_068.tif, test_068.xml).
if DATASET == 'C16':
    with save_shown_figures([os.path.join(FIG_DIR, 'wsi_attention_acmil_test_068.png'),
                             os.path.join(FIG_DIR, 'wsi_attention_acmil_test_067.png')]):
        acmil_attention_visualizations = visualize_wsi_attention(acmil, conf, PROJECT_DIR, acmil_best_state)

### Full experiments for the report (all datasets, several seeds)

The single run above is one seed on one dataset. The report uses `run_experiments.py`, which repeats the identical protocol for every dataset, method and seed (split fixed by `seed: 42` in the YAML; the training seed varies), caches the features in RAM, and saves metrics, learning curves, test predictions and checkpoints under `results/`. `summarize_results.py` then writes the tables and figures that `report/main.tex` includes.

In [ ]:
# Takes roughly 1-2 h on a Colab T4 for 3 datasets x 6 methods x 5 seeds; finished runs are skipped if you rerun.
# !python run_experiments.py --datasets C16 C17 BRACS --models abmil meanpool maxpool acmil acmil_mba acmil_stkim --seeds 0 1 2 3 4 --quiet
# !python summarize_results.py

summary_path = os.path.join(PROJECT_DIR, 'results', 'summary.md')
if os.path.exists(summary_path):
    from IPython.display import Markdown, display
    display(Markdown(open(summary_path).read()))
else:
    print('Run the two commands above first.')